# 26. 사업보고서 청킹 A/B와 근거 검색 비교

**목적:** 25번에서 선택한 B 추출 텍스트를 같은 조건으로 검색할 때, 일반 문장 청킹(A)과 제목 경계 우선 청킹(B) 중 어느 쪽이 정답 근거를 상위 결과에 더 잘 남기는지 비교합니다.

**고정 조건:** 삼성전자·리노공업 2024 사업연도 사업보고서 각 1건, OpenDART 원문에서 직접 확인한 근거 인용문 10개, LlamaIndex `SentenceSplitter` 512 token/overlap 64, 회사별 검색, 동일한 문자 n-gram 검색기. **변수는 제목 경계를 먼저 나누는지 여부뿐**입니다.

**이번에 하지 않는 것:** LLM 답변 생성, 임베딩·벡터 DB·reranker 비교, 최종 청킹 방식 채택. 이 검색기는 청킹 효과를 분리해 보기 위한 고정된 어휘 검색 기준선입니다. 검색기 자체의 품질은 다음 별도 비교 대상입니다.

**지표:** Evidence Coverage = 정답 인용문 전체를 포함하는 청크가 하나라도 있는 질문 비율. Hit@3/5 = 상위 3/5개에 그런 청크가 있는 질문 비율. MRR@5 = 첫 정답 청크 순위의 역수(5위 밖은 0)의 평균. 각 질문에는 정답 인용문 하나만 두므로 여기서 Hit@K는 해당 근거 단위의 Recall@K와 같습니다.

참고: [LlamaIndex Node Parser](https://developers.llamaindex.ai/python/framework/module_guides/loading/node_parsers/), [scikit-learn HashingVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.HashingVectorizer.html), [LlamaIndex Retrieval Evaluation](https://developers.llamaindex.ai/python/framework/understanding/evaluating/evaluating/).

## 1. 환경과 고정 입력

반드시 `conda skn25` 커널에서 실행합니다. API 키는 존재 여부만 확인하고 값이나 요청 URL은 출력하지 않습니다.

In [ ]:
from io import BytesIO
from pathlib import Path
from time import perf_counter
from importlib.metadata import version
import os
import re
import zipfile

from bs4 import BeautifulSoup, Comment, NavigableString, Tag
from dotenv import load_dotenv
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter
from sklearn.feature_extraction.text import HashingVectorizer
import numpy as np
import pandas as pd
import requests

PROJECT_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / '.env').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('프로젝트 최상위 .env를 찾지 못했습니다.')
load_dotenv(PROJECT_ROOT / '.env')
API_KEY = os.getenv('OPENDART_API_KEY')
if not API_KEY:
    raise RuntimeError('.env에 OPENDART_API_KEY가 필요합니다.')
if version('llama-index-core') != '0.14.25':
    raise RuntimeError('이 비교는 llama-index-core 0.14.25에서 작성했습니다. 버전을 확인하세요.')
print('환경 확인 완료 | llama-index-core:', version('llama-index-core'), '| scikit-learn:', version('scikit-learn'))

In [ ]:
REPORTS = {
    '삼성전자': {'rcept_no': '20250311001085', 'xml_name': '20250311001085.xml'},
    '리노공업': {'rcept_no': '20250318000229', 'xml_name': '20250318000229.xml'},
}
GOLD = [
    {'id': 'S01', 'company': '삼성전자', 'question': '삼성전자의 2024년 시설투자 규모는 얼마였나?', 'quote': '53.6조원의 시설투자가 이루어졌습니다.'},
    {'id': 'S02', 'company': '삼성전자', 'question': '삼성전자가 관리하는 주요 재무 위험에는 무엇이 있나?', 'quote': '시장위험, 신용위험, 유동성위험 등을 최소화하는데 중점을 두고'},
    {'id': 'S03', 'company': '삼성전자', 'question': '삼성전자 그룹에 포함된 종속기업 규모는?', 'quote': 'SDC, Harman 산하 종속기업 등 228개의 종속기업으로 구성된 글로벌 전자 기업입니다.'},
    {'id': 'S04', 'company': '삼성전자', 'question': '삼성전자 TV 사업의 2024년까지 판매 1위 지속 기간은?', 'quote': 'TV사업은 2006년부터 2024년까지 19년 연속 판매 1위를 지키고 있습니다.'},
    {'id': 'S05', 'company': '삼성전자', 'question': '삼성전자 모바일사업의 글로벌 출하량 1위 지속 기간은?', 'quote': '모바일사업은 2011년부터 2024년까지 14년 연속 글로벌 출하량 1위를 유지하고 있습니다.'},
    {'id': 'R01', 'company': '리노공업', 'question': '리노공업은 제품별 가격을 산출하기 어려운 이유가 무엇인가?', 'quote': '30,000여종의 제품을 생산하고 있어, 특정 제품에 대한 가격을 개별적으로 산출하기는 어렵습니다.'},
    {'id': 'R02', 'company': '리노공업', 'question': '리노공업은 제품 공급가격을 정할 때 무엇을 고려하는가?', 'quote': '거래처별 공급수량 및 제품의 난이도에 따라 합리적으로 가격을 책정'},
    {'id': 'R03', 'company': '리노공업', 'question': '리노공업이 주시하는 시장위험 요인에는 무엇이 있나?', 'quote': '가. 시장위험과 위험관리 국내외 원자재 가격과 유가의 불안정, 경기 변동, 지정학적 위험'},
    {'id': 'R04', 'company': '리노공업', 'question': '리노공업 연구개발의 기본 개념은 무엇인가?', 'quote': '전 사원의 연구원화'},
    {'id': 'R05', 'company': '리노공업', 'question': '리노공업은 해외 고객을 어떻게 관리하는가?', 'quote': '지역별 혹은 국가별로 현지 대리점을 두어 고객을 관리하고 있으며'},
]
SOURCE_HINTS = {
    'S01': '시설투자 현황', 'S02': '재무위험관리정책', 'S03': '글로벌 기업 구성',
    'S04': 'TV사업', 'S05': '모바일사업', 'R01': '제품 종류',
    'R02': '제품 가격 책정', 'R03': '시장위험과 위험관리',
    'R04': '연구개발', 'R05': '해외영업',
}
for case in GOLD:
    case['rcept_no'] = REPORTS[case['company']]['rcept_no']
    case['source_hint'] = SOURCE_HINTS[case['id']]
display(pd.DataFrame(GOLD))

## 2. 같은 XML을 B 텍스트로 추출

25번과 동일한 접수번호·XML 파일·B 추출 규칙을 사용합니다. 다운로드 실패 시 원래 예외의 URL을 화면에 출력하지 않습니다.

In [ ]:
def download_xml(rcept_no, xml_name):
    try:
        with requests.get('https://opendart.fss.or.kr/api/document.xml',
                          params={'crtfc_key': API_KEY, 'rcept_no': rcept_no},
                          timeout=120, stream=True) as response:
            if response.status_code != 200:
                raise RuntimeError(f'OpenDART 원문 HTTP {response.status_code}')
            chunks, received = [], 0
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                received += len(chunk)
                if received > 60_000_000:
                    raise RuntimeError('원문 ZIP이 60MB를 넘어 다운로드를 중단합니다.')
                chunks.append(chunk)
    except requests.RequestException:
        raise RuntimeError('OpenDART 원문 요청 실패: 연결·인증·응답 상태를 확인하세요. 요청 URL은 출력하지 않습니다.') from None
    payload = b''.join(chunks)
    if not zipfile.is_zipfile(BytesIO(payload)):
        raise RuntimeError('OpenDART 응답이 ZIP 형식이 아닙니다. 키·접수번호·API 상태를 확인하세요.')
    with zipfile.ZipFile(BytesIO(payload)) as archive:
            if xml_name not in archive.namelist():
                raise RuntimeError('25번에서 선택한 XML 파일을 이번 ZIP에서 찾지 못했습니다.')
            info = archive.getinfo(xml_name)
            if info.file_size > 60_000_000 or sum(item.file_size for item in archive.infolist()) > 120_000_000:
                raise RuntimeError('ZIP 압축 해제 크기가 실험 한도를 넘습니다.')
            return archive.read(xml_name)

def parse_source_and_b(xml_bytes):
    soup = BeautifulSoup(xml_bytes, 'lxml-xml')
    if soup.find() is None:
        raise RuntimeError('XML에서 태그를 찾지 못했습니다.')
    source_text = re.sub(r'\s+', ' ', soup.get_text(' ', strip=True)).strip()
    parts = []
    for node in soup.descendants:
        if isinstance(node, Tag) and node.name.upper() == 'TR':
            parts.append('\n')
        if not isinstance(node, NavigableString) or isinstance(node, Comment):
            continue
        value = re.sub(r'\s+', ' ', str(node)).strip()
        if not value:
            continue
        ancestors = {tag.name.upper() for tag in node.parents if tag.name}
        if ancestors & {'TITLE', 'SUBTITLE'}:
            parts.append('\n## ' + value + '\n')
        elif ancestors & {'TD', 'TH'}:
            parts.append(value + ' | ')
        else:
            parts.append(value + ' ')
    b_text = re.sub(r' *\n *', '\n', ''.join(parts)).strip()
    return source_text, b_text

In [ ]:
source_texts, texts = {}, {}
for company, report in REPORTS.items():
    xml_bytes = download_xml(report['rcept_no'], report['xml_name'])
    source_texts[company], texts[company] = parse_source_and_b(xml_bytes)
    print(company, '| 접수번호:', report['rcept_no'], '| B 글자 수:', len(texts[company]))
    del xml_bytes

gold_counts = [(case['id'], source_texts[case['company']].count(case['quote']),
                texts[case['company']].count(case['quote'])) for case in GOLD]
display(pd.DataFrame(gold_counts, columns=['case_id', 'parsed_XML_occurrences', 'B_occurrences']))
if any(source_count != 1 or b_count != 1 for _, source_count, b_count in gold_counts):
    raise RuntimeError('정답 인용문이 XML 파싱 텍스트와 B에서 각각 한 번씩 발견되지 않았습니다. 평가를 중단합니다.')

## 3. 청킹 후보만 변경

A는 B 텍스트 전체를 문장 기준으로 나눕니다. B는 `##` 제목에서 먼저 구간을 끊은 뒤 **같은** `SentenceSplitter`를 적용합니다. 제목을 모든 청크에 새로 주입하지는 않습니다. 따라서 이번 차이는 제목 경계를 넘겨 자를 수 있는지 여부입니다.

In [ ]:
SPLITTER = SentenceSplitter(chunk_size=512, chunk_overlap=64)

def chunk_text(text, candidate):
    if candidate == 'A_full':
        sections = [text]
    elif candidate == 'B_headings':
        sections = [part for part in re.split(r'(?=^## )', text, flags=re.MULTILINE) if part.strip()]
    else:
        raise ValueError('알 수 없는 청킹 후보입니다.')
    chunks = []
    for section in sections:
        nodes = SPLITTER.get_nodes_from_documents([Document(text=section)], show_progress=False)
        chunks.extend(node.text.strip() for node in nodes if node.text.strip())
    return chunks, len(sections)

## 4. 동일 검색기와 평가

문자 2~4-gram `HashingVectorizer`는 학습·IDF 추정이 없어 두 후보의 검색 표현을 같은 규칙으로 고정할 수 있습니다. 이 점수는 **어휘 검색 기준선에서의 청킹 효과**이며 임베딩 검색 결과로 일반화하지 않습니다.

In [ ]:
VECTORIZER = HashingVectorizer(analyzer='char', ngram_range=(2, 4),
                               n_features=2**18, norm='l2', alternate_sign=False, dtype=np.float32)
indexes, build_rows = {}, []
for candidate in ['A_full', 'B_headings']:
    for company, text in texts.items():
        started = perf_counter()
        chunks, section_count = chunk_text(text, candidate)
        matrix = VECTORIZER.fit_transform(chunks)
        elapsed = perf_counter() - started
        indexes[(candidate, company)] = {'chunks': chunks, 'matrix': matrix}
        sparse_bytes = matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes
        build_rows.append({'candidate': candidate, 'company': company, 'sections': section_count,
                           'chunks': len(chunks), 'median_chunk_chars': int(np.median([len(x) for x in chunks])),
                           'build_seconds': round(elapsed, 2), 'sparse_matrix_MB': round(sparse_bytes / 1024**2, 2)})
display(pd.DataFrame(build_rows))

In [ ]:
def compact(value):
    return re.sub(r'\s+', '', value)

result_rows = []
for candidate in ['A_full', 'B_headings']:
    for case in GOLD:
        index = indexes[(candidate, case['company'])]
        chunks, matrix = index['chunks'], index['matrix']
        evidence = compact(case['quote'])
        relevant = [evidence in compact(chunk) for chunk in chunks]
        started = perf_counter()
        query = VECTORIZER.transform([case['question']])
        scores = (matrix @ query.T).toarray().ravel()
        order = np.argsort(-scores, kind='stable')
        ties_through_rank_5 = sum(scores[order[i]] == scores[order[i + 1]]
                         for i in range(min(5, len(order) - 1)))
        latency_ms = (perf_counter() - started) * 1000
        rank = next((position for position, chunk_id in enumerate(order, start=1) if relevant[chunk_id]), None)
        result_rows.append({'candidate': candidate, 'case_id': case['id'], 'company': case['company'],
                            'evidence_coverage': any(relevant), 'first_evidence_rank': rank,
                            'hit_at_3': rank is not None and rank <= 3,
                            'hit_at_5': rank is not None and rank <= 5,
                            'rr_at_5': 1 / rank if rank is not None and rank <= 5 else 0.0,
                            'query_ms': round(latency_ms, 2),
                            'ties_through_rank_5': ties_through_rank_5,
                            'top_1_excerpt': chunks[order[0]][:180]})
results = pd.DataFrame(result_rows)
display(results[['candidate', 'case_id', 'evidence_coverage', 'first_evidence_rank', 'hit_at_3', 'hit_at_5', 'ties_through_rank_5', 'query_ms']])

In [ ]:
summary = results.groupby('candidate', as_index=False).agg(
    cases=('case_id', 'count'),
    evidence_coverage=('evidence_coverage', 'mean'),
    hit_at_3=('hit_at_3', 'mean'),
    hit_at_5=('hit_at_5', 'mean'),
    mrr_at_5=('rr_at_5', 'mean'),
    median_query_ms=('query_ms', 'median'),
)
display(summary)
misses = results[~results['hit_at_5']]
if not misses.empty:
    print('상위 5개에서 근거를 놓친 사례 — 실패 원인 확인용 첫 검색 결과:')
    display(misses[['candidate', 'case_id', 'first_evidence_rank', 'top_1_excerpt']])
else:
    print(f'두 후보 모두 {len(GOLD)}개 사례에서 Hit@5를 충족했습니다. 이 경우 청킹 우위를 결론내리지 않습니다.')

## 결과를 읽는 기준

- 먼저 `parsed_XML_occurrences`와 `B_occurrences`가 모두 1인지 봅니다. 0이나 중복이 있으면 비교를 중단합니다. 두 텍스트는 동일 XML 파서에서 출발하므로 이 검사는 원본 바이트 전체의 무손실 증명이 아닙니다.
- `evidence_coverage=False`는 정답 문구가 청크 경계에서 갈라졌거나 청킹 중 사라졌다는 뜻입니다. `first_evidence_rank`는 **전체 후보 청크**에서 처음 정답 문구가 나온 순위이며, 5위 밖이면 MRR@5는 0입니다.
- 후보별 Hit@3/5·MRR@5를 우선 보되, 질문 10개·문서 2건뿐이라 작은 차이는 확정하지 않습니다. 청크 수·행렬 크기·인덱스 생성 시간도 함께 봅니다. `ties_through_rank_5`는 1~5위 내부와 5·6위 경계의 점수 동점 개수이며, 동점은 원문 청크 순서로 결정합니다.
- 검색기는 고정된 어휘 방식입니다. **임베딩·하이브리드·reranker의 우열**은 이 노트북으로 판단할 수 없습니다. 개발셋에서 선택한 청킹 후보는 더 넓은 질문·문서와 별도 holdout에서 다시 확인해야 합니다.